# 10 – Results: Modelo Final

**Proyecto:** Análisis y predicción del subempleo por insuficiencia de horas en el Perú – EPEN 2024  
**Target:** `target_subempleo_horas` (1 = subempleado por horas · 0 = no subempleado)  
**Objetivo:** Documentar, serializar y validar el modelo final seleccionado.

**Modelo seleccionado:** `Logistic Regression (class_weight='balanced')`  
**Criterio de selección:** Mayor F1-score clase 1 (0.475) y ROC-AUC (0.697) en el conjunto de prueba, según `data/results/model_comparison.csv`.

> **Prerequisito:** Ejecuta `06_feature_selection/selected_variables.ipynb` y `07_modelling/01_baseline_model.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score
import joblib
import json
from pathlib import Path
from datetime import datetime

# ── Rutas ──────────────────────────────────────────────────────────────────────
SEL_DIR     = Path('../data/selected')
MODEL_DIR   = Path('../models')
RESULTS_DIR = Path('../data/results')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET      = 'target_subempleo_horas'
CLASS_NAMES = ['No subempleado por horas', 'Subempleado por horas']

# ── Carga de datos reales (sin fallback sintético) ─────────────────────────────
required = {
    'X_train': SEL_DIR / 'X_train_selected.csv',
    'X_test':  SEL_DIR / 'X_test_selected.csv',
    'y_train': SEL_DIR / 'y_train_selected.csv',
    'y_test':  SEL_DIR / 'y_test_selected.csv',
}
for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Archivo requerido no encontrado: {path}\n"
            "Ejecuta primero: 06_feature_selection/selected_variables.ipynb"
        )

X_train = pd.read_csv(required['X_train'])
X_test  = pd.read_csv(required['X_test'])

def load_target(path, name):
    df = pd.read_csv(path)
    if TARGET in df.columns:
        return df[TARGET].reset_index(drop=True)
    if df.shape[1] == 1:
        return df.iloc[:, 0].reset_index(drop=True)
    raise ValueError(f"No se encontró '{TARGET}' en {name}")

y_train = load_target(required['y_train'], 'y_train')
y_test  = load_target(required['y_test'],  'y_test')

selected = X_train.columns.tolist()

print(f'X_train : {X_train.shape}  |  X_test : {X_test.shape}')
print(f'y_train clase 1: {y_train.mean():.2%}  |  y_test clase 1: {y_test.mean():.2%}')
print('Datos cargados correctamente.')


## 1. Carga del modelo ganador

El modelo ganador ya fue entrenado y serializado en `07_modelling/01_baseline_model.ipynb`.  
Se carga directamente y se serializa como `final_model_pipeline.pkl` para entrega.


In [ ]:
# Cargar el modelo ganador: Logistic Regression (class_weight='balanced')
# Pipeline: StandardScaler → LogisticRegression
# Entrenado en: 07_modelling/01_baseline_model.ipynb
model_path = MODEL_DIR / 'logistic_regression_balanced.pkl'
if not model_path.exists():
    raise FileNotFoundError(
        f"Modelo no encontrado: {model_path}\n"
        "Ejecuta primero: 07_modelling/01_baseline_model.ipynb"
    )

final_pipeline = joblib.load(model_path)
print(f'Modelo cargado: {model_path.name}')
print(f'Tipo: {type(final_pipeline).__name__}')
if hasattr(final_pipeline, 'named_steps'):
    print(f'Pasos: {list(final_pipeline.named_steps.keys())}')


## 2. Validación final en conjunto de prueba

In [ ]:
y_pred_final = final_pipeline.predict(X_test)
y_prob_final = final_pipeline.predict_proba(X_test)[:, 1]
auc_final    = roc_auc_score(y_test, y_prob_final)

print('=== Evaluación Final – Logistic Regression (class_weight="balanced") ===\n')
print(classification_report(y_test, y_pred_final, target_names=CLASS_NAMES))
print(f'ROC-AUC Final: {auc_final:.4f}')


## 3. Serialización del modelo

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

# Serializar como final_model_pipeline.pkl
final_path = MODEL_DIR / 'final_model_pipeline.pkl'
joblib.dump(final_pipeline, final_path)
print(f'Modelo final guardado: {final_path}')

# Metadata del modelo
metadata = {
    'proyecto':            'Análisis y predicción del subempleo por insuficiencia de horas en el Perú',
    'fuente_datos':        'INEI – EPEN 2024',
    'fecha_entrenamiento': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'algoritmo':           'LogisticRegression',
    'class_weight':        'balanced',
    'preprocesamiento':    'StandardScaler (dentro del pipeline)',
    'variable_objetivo':   TARGET,
    'clase_positiva':      '1 = subempleado por insuficiencia de horas',
    'clase_negativa':      '0 = no subempleado por horas',
    'variables_entrada':   selected,
    'n_features':          len(selected),
    'n_train':             int(X_train.shape[0]),
    'n_test':              int(X_test.shape[0]),
    'metricas_test': {
        'accuracy':    round(accuracy_score(y_test, y_pred_final),  4),
        'precision_1': round(precision_score(y_test, y_pred_final, zero_division=0), 4),
        'recall_1':    round(recall_score(y_test, y_pred_final, zero_division=0), 4),
        'f1_1':        round(f1_score(y_test, y_pred_final, zero_division=0), 4),
        'roc_auc':     round(auc_final, 4),
    },
    'criterio_seleccion':  'Mayor F1-score clase 1 y ROC-AUC entre todos los modelos evaluados',
}

meta_path = MODEL_DIR / 'model_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f'Metadata guardada: {meta_path}')
print(f"\nMétricas finales: Accuracy={metadata['metricas_test']['accuracy']} | "
      f"F1={metadata['metricas_test']['f1_1']} | ROC-AUC={metadata['metricas_test']['roc_auc']}")


## 4. Ejemplo de predicción con datos reales del conjunto de prueba

Se seleccionan 5 observaciones reales del test set para ilustrar las predicciones del modelo.


In [ ]:
# Seleccionar 5 observaciones reales del test set (2 positivos + 3 negativos si hay)
pos_idx = y_test[y_test == 1].index[:2].tolist()
neg_idx = y_test[y_test == 0].index[:3].tolist()
sample_idx = pos_idx + neg_idx

muestra = X_test.loc[sample_idx].reset_index(drop=True)
y_real  = y_test.loc[sample_idx].reset_index(drop=True)

predicciones  = final_pipeline.predict(muestra)
probabilidades = final_pipeline.predict_proba(muestra)[:, 1]

resultado = pd.DataFrame({
    'y_real':              y_real.values,
    'etiqueta_real':       y_real.map({0: 'No subempleado por horas', 1: 'Subempleado por horas'}),
    'prediccion':          predicciones,
    'etiqueta_predicha':   pd.Series(predicciones).map({0: 'No subempleado por horas', 1: 'Subempleado por horas'}),
    'prob_subempleado':    probabilidades.round(3),
    'correcto':            (predicciones == y_real.values),
})

print('Predicciones sobre observaciones reales del conjunto de prueba:')
resultado
